# Configuração única do token Hugging Face
Na interface nativa do Colab, escolha usar o Secret `HF_TOKEN`. No kernel do VS Code, escolha entrada oculta e informe o valor do token; não tente acessar Secrets por esse kernel. O OpenFake é público e esta configuração é opcional. O token será salvo apenas cifrado no Drive.

In [1]:
%pip install cryptography
from google.colab import drive, userdata
from pathlib import Path
drive.mount('/content/drive')
target = Path('/content/drive/MyDrive/TCC_Final/credenciais/huggingface.enc')
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists():
    raise FileExistsError(f'Credencial existente preservada: {target}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import base64, getpass, json, os
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
use_secret = input('Executando na interface nativa do Colab? Usar Secret HF_TOKEN? [s/N]: ').strip().lower() == 's'
if use_secret:
    hf_token = userdata.get('HF_TOKEN')
else:
    hf_token = getpass.getpass('HF_TOKEN (não será exibido): ')
if not hf_token:
    raise RuntimeError('HF_TOKEN não informado')
password = getpass.getpass('Crie uma senha forte para desbloquear no VS Code: ')
confirmation = getpass.getpass('Repita a senha: ')
if password != confirmation or len(password) < 5:
    raise ValueError('As senhas devem coincidir e ter pelo menos 12 caracteres')
salt, iterations = os.urandom(16), 600000
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=iterations).derive(password.encode()))
ciphertext = Fernet(key).encrypt(hf_token.encode()).decode()
with target.open('x', encoding='utf-8') as output:
    json.dump({'version': 1, 'iterations': iterations, 'salt': base64.b64encode(salt).decode(), 'ciphertext': ciphertext}, output)
os.chmod(target, 0o600)
del hf_token, password, confirmation, key, ciphertext
print('Credencial Hugging Face cifrada no Drive.')

Credencial Hugging Face cifrada no Drive.
